<!--
Copyright Amazon.com, Inc. or its affiliates. All Rights Reserved.
SPDX-License-Identifier: MIT-0
-->


In [ ]:
%store -r

In [ ]:
%store

# Lab 2: Supervised Fine-Tuning (SFT)

## Create the Evaluator

Before training anything, you build the thing that measures it.

The evaluator is a Lambda function that scores a generated SQL query by **executing it against the live Aurora database** and comparing the rows it returns against the rows the reference query returns. Not string similarity — actual results.

That design choice is what makes the rest of the workshop work:

- In **this lab** it is the evaluation metric that tells you whether SFT helped.
- In **Lab 3** the exact same function becomes the *reward function* for reinforcement learning. RLVR needs a programmatic, trustworthy answer to "was that output correct?", and executing the query is that answer.
- In **Lab 4** it scores Claude Haiku and Claude Sonnet too, so the frontier comparison uses the identical scorer.

One function, three roles. Build it once, carefully.

The next cell makes a `lambda/` directory to package it from.

In [ ]:
!mkdir lambda

### How the evaluator actually scores a query

This is the most important cell in the workshop to understand, because **every number you see in Labs 2, 3 and 4 comes out of it**. Read the metric definitions before you run it — the differences between them are subtle and they explain a lot of otherwise confusing results.

For each sample, the function pulls out the model's SQL and the reference SQL (stripping any ` ```sql ` fencing the model wrapped it in), executes both through the RDS Data API, and computes four numbers.

| Metric | Definition | The catch |
|---|---|---|
| `execution_success` | 1.0 if the generated SQL ran without error, else 0.0 | **Says nothing about correctness.** `SELECT 1` scores 1.0. A syntactically valid query returning entirely the wrong rows still scores 1.0. This measures "is it SQL", not "is it right" |
| `execution_accuracy` | 1.0 only if `completion_records == reference_records` | This is a Python list comparison, so it is **order-sensitive**. A logically correct query that returns the same rows in a different order scores **0**. Strictest metric here, and it under-reports genuine correctness |
| `result_set_f1` | Set F1 over result rows, where each row is hashed as `tuple(sorted(row.items()))` | Order-*insensitive*, which fixes the problem above. But **column names are part of the hash**, so `SUM(price) AS total` vs `SUM(price) AS sum` produces identical numbers with different keys and scores **0**. Also gives partial credit — 8 of 10 correct rows is not zero |
| `aggregate_reward` | `0.3 × execution_success + 0.7 × result_set_f1` | **This is the number that matters.** Read the weighting note below |

### The 0.3 / 0.7 weighting

```
aggregate_reward = 0.3 * execution_success + 0.7 * result_set_f1
```

Two things follow from this, and both surprise people:

1. **There is a floor of 0.30 for anything that merely parses.** A model that emits `SELECT * FROM product_sales` for every question scores 0.30. So do not read 0.30 as "30% correct" — read it as "valid SQL, wrong answer". This is why the base model in the comparison table sits around 0.43 rather than near zero.
2. **Correctness is weighted more than validity, but validity is not free.** Why include `execution_success` at all if you care about correctness? Because of Lab 3. In reinforcement learning the model needs a *gradient*, and a pure-correctness reward gives every wrong answer the same score of zero — malformed gibberish and a query that is one join away from right look identical. The 0.3 term is a stepping stone: it rewards "you produced runnable SQL" as progress worth having on the way to being correct.

`result_set_f1` rather than `execution_accuracy` carries the 0.7 for the same reason. Exact ordered match is all-or-nothing; F1 gives partial credit and therefore a smoother reward surface to climb.

::alert[This weighting is a design decision, not a standard. It is arguably the highest-leverage knob in the whole workshop and you could reasonably choose differently — see the ablation discussion in Lab 3.]{type="info"}

### One naming trap worth knowing now

The function returns the aggregate under the key **`aggregate_reward_score`**, while the four individual metrics come back in `metrics_list` under their plain names. In MLflow it can also appear as `eval/0/aggregate_reward_score` depending on which harness logged it. That inconsistency is exactly why Lab 4's comparison code carries a `METRIC_MAP` with fallback key names — if you ever see a blank column in that table, this is the first place to look.

### Also note

- The function handles **both** payload shapes: SFT evaluation sends `completion` + `reference_answer`, while RLVR sends `response` + `reward_model.ground_truth`. That dual support is what lets one Lambda serve both labs.
- It is chatty — lots of `print()` of full SQL and result sets. Those land in CloudWatch Logs, and they are genuinely useful when a score looks wrong and you want to see what the model actually emitted.
- Every scored sample costs **two** database round-trips (candidate and reference). With `rollout_n = 32` in Lab 3, that adds up quickly.

In [ ]:
%%writefile lambda/evaluator.py
# Copyright Amazon.com, Inc. or its affiliates. All Rights Reserved.
# SPDX-License-Identifier: MIT-0
import json
import os
import re
import boto3
from typing import Dict, Any, List, Tuple

rds_data = boto3.client('rds-data')

CLUSTER_ARN = os.environ.get('CLUSTER_ARN', '')
SECRET_ARN = os.environ.get('SECRET_ARN', '')
DATABASE = os.environ.get('DATABASE', '')


# =========================================================================================
# Helper: Extract SQL queries from sample
# =========================================================================================
def extract_sql_query(sample: Dict[str, Any]) -> Tuple[str, str]:
    """
    Extract valid SQL statements from the sample payload.
    Supports both SFT eval format and RLVR format:
      - SFT:  sample['completion'] + sample['reference_answer']
      - RLVR: sample['response']   + sample['reward_model']['ground_truth']
    Strips any '```sql' or '```' markdown fencing found.
    """
    def clean_sql(text: str) -> str:
        if not text:
            return ""
        text = text.strip()
        text = re.sub(r'```sql\s*', '', text)
        text = re.sub(r'```\s*', '', text)
        return text.strip()

    # Model completion: RLVR sends 'response', SFT eval sends 'completion'
    completion = sample.get('response', '') or sample.get('completion', '')

    # Ground truth: RLVR sends 'reward_model.ground_truth', SFT eval sends 'reference_answer'
    reward_model = sample.get('reward_model', {})
    if isinstance(reward_model, dict):
        reference_answer = reward_model.get('ground_truth', '')
    else:
        reference_answer = ''

    if not reference_answer:
        reference_answer = sample.get('reference_answer', '')
        if isinstance(reference_answer, dict):
            reference_answer = reference_answer.get('text', '')

    return clean_sql(completion), clean_sql(reference_answer)


# =========================================================================================
# Helper: Run SQL query against Aurora via RDS Data API
# =========================================================================================
def call_db(sql: str) -> Dict[str, Any]:
    """
    Execute a SQL query against Aurora PostgreSQL using the RDS Data API.
    """
    if not sql:
        return {'success': False, 'error': 'Empty SQL query', 'records': []}

    try:
        response = rds_data.execute_statement(
            resourceArn=CLUSTER_ARN,
            secretArn=SECRET_ARN,
            database=DATABASE,
            sql=sql,
            includeResultMetadata=True
        )

        columns = [col['name'] for col in response.get('columnMetadata', [])]
        rows = []
        for record in response.get('records', []):
            row = {}
            for i, field in enumerate(record):
                if 'isNull' in field and field['isNull']:
                    row[columns[i]] = None
                elif 'stringValue' in field:
                    row[columns[i]] = field['stringValue']
                elif 'longValue' in field:
                    row[columns[i]] = field['longValue']
                elif 'doubleValue' in field:
                    row[columns[i]] = field['doubleValue']
                elif 'booleanValue' in field:
                    row[columns[i]] = field['booleanValue']
                else:
                    row[columns[i]] = str(field)
            rows.append(row)

        return {'success': True, 'records': rows, 'error': None}

    except Exception as e:
        return {'success': False, 'records': [], 'error': str(e)}


# =========================================================================================
# Helper: Compute result set precision, recall, and F1
# =========================================================================================
def compute_result_set_metrics(completion_records: List[Dict], reference_records: List[Dict]) -> Dict[str, float]:
    """
    Compare completion and reference result sets using set-based precision/recall/F1.
    Rows are converted to frozensets for order-independent comparison.
    """
    def row_to_hashable(row: Dict) -> tuple:
        return tuple(sorted(row.items()))

    completion_set = set(row_to_hashable(r) for r in completion_records)
    reference_set = set(row_to_hashable(r) for r in reference_records)

    if not reference_set and not completion_set:
        return {'precision': 1.0, 'recall': 1.0, 'f1': 1.0}

    if not reference_set:
        return {'precision': 0.0, 'recall': 1.0, 'f1': 0.0}

    if not completion_set:
        return {'precision': 1.0, 'recall': 0.0, 'f1': 0.0}

    true_positives = len(completion_set & reference_set)
    precision = true_positives / len(completion_set) if completion_set else 0.0
    recall = true_positives / len(reference_set) if reference_set else 0.0
    f1 = (2 * precision * recall) / (precision + recall) if (precision + recall) > 0 else 0.0

    return {'precision': precision, 'recall': recall, 'f1': f1}


# =========================================================================================
# Reward function
# =========================================================================================
def reward_function(sample: Dict[str, Any], index: int) -> Dict[str, Any]:
    """
    Args:
        sample: Dictionary containing model completion and reference SQL
        index: Sample index in batch

    Returns:
        Dictionary with reward scores and metrics
    """
    completion_sql, reference_sql = extract_sql_query(sample)

    print("Completion SQL:\n\n", completion_sql, "\n\n\n")
    print("Reference SQL:\n\n", reference_sql, "\n\n\n")

    completion_result = call_db(completion_sql)
    print("Completion Result:", completion_result, "\n\n\n")
    reference_result = call_db(reference_sql)
    print("Reference Result:", reference_result, "\n\n\n")

    execution_success = 1.0 if completion_result['success'] else 0.0

    execution_accuracy = 0.0
    if completion_result['success'] and reference_result['success']:
        if completion_result['records'] == reference_result['records']:
            execution_accuracy = 1.0

    if completion_result['success'] and reference_result['success']:
        set_metrics = compute_result_set_metrics(
            completion_result['records'],
            reference_result['records']
        )
    else:
        set_metrics = {'precision': 0.0, 'recall': 0.0, 'f1': 0.0}

    aggregate_reward = 0.3 * execution_success + 0.7 * set_metrics['f1']

    metrics = [
        {
            'name': 'execution_success',
            'value': float(execution_success),
            'type': 'Metric'
        },
        {
            'name': 'execution_accuracy',
            'value': float(execution_accuracy),
            'type': 'Metric'
        },
        {
            'name': 'result_set_recall',
            'value': float(set_metrics['recall']),
            'type': 'Metric'
        },
        {
            'name': 'result_set_precision',
            'value': float(set_metrics['precision']),
            'type': 'Metric'
        },
        {
            'name': 'result_set_f1',
            'value': float(set_metrics['f1']),
            'type': 'Reward'
        }
    ]

    return {
        'id': str(sample.get('sample_id', f'sample-{index:03d}')),
        'aggregate_reward_score': float(aggregate_reward),
        'metrics_list': metrics,
        'completion_db_output': completion_result,
        'reference_db_output': reference_result
    }


def lambda_handler(event: Dict[str, Any], context: Any) -> Dict[str, Any]:
    """
    AWS Lambda Handler for reward function
    """
    print(event)
    try:
        batch = event.get('input', event) if isinstance(event, dict) else event
        if 'batch' in event:
            batch = event.get('batch', [])
        elif 'body' in event:
            body = json.loads(event.get('body', '{}'))
            batch = body.get('batch', [])

        if not batch:
            return {"error": "Missing or empty batch"}

        results = []
        for i, sample in enumerate(batch):
            print(sample)
            try:
                result = reward_function(sample, i)
                results.append(result)
            except Exception as e:
                return {"error": str(e)}

        return {
            'statusCode': 200,
            'headers': {'Content-Type': 'application/json'},
            'body': json.dumps(results)
        }
    except Exception as e:
        return {
            'statusCode': 400,
            'body': json.dumps({"error": str(e)})
        }

### Package it for Lambda

Zip the `lambda/` directory and upload it to S3. The file is written flat inside the archive (`evaluator.py` at the root, not `lambda/evaluator.py`) because Lambda resolves the handler path `evaluator.lambda_handler` relative to the archive root.

No dependencies to bundle — `boto3` is already in the Lambda Python runtime.

In [ ]:
from zipfile import ZipFile
from sagemaker.core.s3 import S3Uploader

with ZipFile("evaluator.zip", "w") as zip_file:
    import os
    for root, dirs, files in os.walk("lambda"):
        for file in files:
            file_path = os.path.join(root, file)
            arcname = os.path.relpath(file_path, "lambda")
            zip_file.write(file_path, arcname=arcname)

S3Uploader.upload('evaluator.zip', desired_s3_uri=f's3://{DEFAULT_BUCKET}/lambdas')

## Deploy the Lambda

We package the evaluator as an AWS Lambda function. Lambda is ideal here because it is serverless (no infrastructure to manage), scales automatically when SageMaker sends batches of samples for scoring, and the same deployment handles both SFT evaluation and RLVR reward computation.

Three details in the cell below matter later:

- **The Aurora connection details are passed as environment variables**, not baked into the code. That is how the same zip file works against any cluster.
- **`LAMBDA_ROLE` comes from Lab 0.** It is the role CloudFormation created with permission to read the secret and call the Data API — the Lambda has database access, the notebook does not have the password.
- **`LAMBDA_FUNCTION_NAME = 'workshop_evaluator'`** is `%store`d at the end of this notebook, because Lab 4 invokes this function *directly* (rather than through SageMaker) to score the Claude baselines.

::alert[This cell calls `create_function`, so re-running it after a successful run fails with `ResourceConflictException` — the function already exists. That is harmless. If you need to change the evaluator code, either delete the function first or switch to `update_function_code`.]{type="warning"}

In [ ]:
import boto3

client = boto3.client('lambda')

# Aurora connection details reused from Lab 0, not reconstructed here. Lab 0
# built these three values, verified them with a live SELECT 1, and stored
# them; this notebook's first cell restores them via %store -r. Passing them
# straight through means there is a single source of truth (no 'dev-' prefix to
# keep in sync across notebooks) and the ARNs handed to the evaluator Lambda are
# already proven to resolve. If NameError fires here, run 00-setup.ipynb first.
CLUSTER_ARN = AURORA_CLUSTER_ARN
SECRET_ARN = AURORA_SECRET_ARN
DATABASE = AURORA_DB_NAME

# Lambda function name -- Lab 4 invokes this function directly to score
# Bedrock baselines, so it is persisted with the other shared variables.
LAMBDA_FUNCTION_NAME = 'workshop_evaluator'

ENV_VARS = {
    'CLUSTER_ARN': CLUSTER_ARN,
    'SECRET_ARN': SECRET_ARN,
    'DATABASE': DATABASE,
}
CODE = {'S3Bucket': DEFAULT_BUCKET, 'S3Key': 'lambdas/evaluator.zip'}

# Create the function, or update it if it already exists. Making this cell
# idempotent means you can safely re-run it -- e.g. after editing the evaluator
# code above and re-uploading the zip -- without hitting ResourceConflictException
# and without the silent no-op that a bare create-and-ignore would give you.
# No KMSKeyArn is passed: the evaluator needs no customer-managed key, so Lambda
# uses the default AWS-owned key; omitting the argument requests that default.
try:
    response = client.create_function(
        FunctionName=LAMBDA_FUNCTION_NAME,
        Role=LAMBDA_ROLE,
        Handler='evaluator.lambda_handler',
        Runtime='python3.12',
        Code=CODE,
        Environment={'Variables': ENV_VARS},
        Timeout=60,
        MemorySize=256,
    )
    print(f"Created Lambda function: {LAMBDA_FUNCTION_NAME}")
except client.exceptions.ResourceConflictException:
    # Already exists from a previous run. Push both the latest code and the
    # latest configuration so a re-run actually applies any changes rather than
    # silently keeping the old function.
    print(f"Function {LAMBDA_FUNCTION_NAME} already exists -- updating code and configuration.")
    client.update_function_code(FunctionName=LAMBDA_FUNCTION_NAME, **CODE)
    client.get_waiter('function_updated_v2').wait(FunctionName=LAMBDA_FUNCTION_NAME)
    client.update_function_configuration(
        FunctionName=LAMBDA_FUNCTION_NAME,
        Role=LAMBDA_ROLE,
        Handler='evaluator.lambda_handler',
        Runtime='python3.12',
        Environment={'Variables': ENV_VARS},
        Timeout=60,
        MemorySize=256,
    )
    client.get_waiter('function_updated_v2').wait(FunctionName=LAMBDA_FUNCTION_NAME)
    response = client.get_function(FunctionName=LAMBDA_FUNCTION_NAME)['Configuration']

# Expose the ARN uniformly regardless of which branch ran -- the next cell
# (Evaluator.create) reads response['FunctionArn'].
print(f"Function ARN: {response['FunctionArn']}")

In [ ]:
# Verify the evaluator deployed and is ready to invoke before moving on.
#
# create_function returns as soon as the function is registered, which can be
# before its State flips from "Pending" to "Active". The waiter blocks until it
# is genuinely invokable, so a green result here means Part 2 can proceed.
waiter = client.get_waiter("function_active_v2")
waiter.wait(FunctionName=LAMBDA_FUNCTION_NAME)

info = client.get_function(FunctionName=LAMBDA_FUNCTION_NAME)["Configuration"]

print("Evaluator Lambda deployed successfully.")
print(f"  Function name: {info['FunctionName']}")
print(f"  Function ARN:  {info['FunctionArn']}")
print(f"  State:         {info['State']}")
print(f"  Runtime:       {info['Runtime']}")
print("\nReady to register as a SageMaker Evaluator (next cell).")

### Register the Lambda as a SageMaker Evaluator

The Lambda exists, but SageMaker does not know about it yet. This cell registers it in the AI Registry as an `Evaluator` of type `REWARD_FUNCTION`, which returns an ARN.

That ARN is the handle everything downstream uses:

- The `CustomScorerEvaluator` later in this notebook passes it to score the SFT model
- **Lab 3 passes it to `RLVRTrainer` as the reward function**, which is where `REWARD_FUNCTION` earns its name — the trainer calls this Lambda for every rollout of every step

It is `%store`d as `EVALUATOR_ARN` at the end of the notebook so Lab 3 can pick it up.

In [ ]:
from sagemaker.ai_registry.evaluator import Evaluator
from sagemaker.ai_registry.evaluator import REWARD_FUNCTION
evaluator = Evaluator.create(
    name = "workshop-evaluator",
    source=response['FunctionArn'],
    type=REWARD_FUNCTION
)

print(f"Evaluator registered: {evaluator.name}")
print(f"  ARN: {evaluator.arn}")
print("This ARN is used to score the SFT model below, and is %store'd as")
print("EVALUATOR_ARN at the end of the notebook for Lab 3's reward function.")

# Train with SFT

SFT (Supervised Fine-Tuning) teaches the model by imitation. You show it your `{prompt, completion}` pairs from Lab 1 and it learns to produce completions that look like yours — your schema, your column names, your conventions, your terse no-markdown output style.

## Why LoRA and not full fine-tuning

`training_type=TrainingType.LORA` freezes the base model's weights entirely and trains small low-rank adapter matrices instead — roughly 1% of the parameter count. For this task that is not a compromise, it is the better choice:

- **It fits.** Full fine-tuning of a 3B model needs memory for weights, gradients and optimizer state — several times the model size. LoRA needs a fraction of that, so it trains on a much smaller (and much cheaper) GPU.
- **It overfits less.** You have ~200 training examples. Letting all 3 billion parameters move would let the model memorize them, and it would happily forget general SQL competence in the process. Freezing the base preserves what the model already knows and only adds a small correction.
- **It is portable.** The adapter is a few tens of MB, not gigabytes.

The honest trade-off: LoRA has less capacity than full fine-tuning, so if you needed the model to learn genuinely new *capabilities* rather than a new style and schema, you would eventually hit its ceiling. Teaching it your schema is well within it.

## Why these hyperparameters

| Setting | Value | Reasoning |
|---|---|---|
| `learning_rate` | `1e-5` | Low enough to nudge behavior without overwriting the SQL knowledge the model came with. Too high and you get catastrophic forgetting — a model that emits your column names in queries that no longer make sense |
| `max_epochs` | `8` | With only ~200 examples, one pass is nowhere near enough signal to shift behavior. Eight passes is enough to learn the schema and format. Push it much higher and the model starts memorizing individual examples rather than generalizing |

Both are deliberately conservative. Note the contrast coming in Lab 3: RLVR uses `1e-6`, ten times lower again, because RL gradient estimates are far noisier than supervised ones.

## Where the metrics go

`mlflow_experiment_name='sft-training'` is the whole MLflow integration. Pass that plus `mlflow_resource_arn` and the trainer streams loss and learning-rate curves to your tracking server automatically — nothing to instrument, no logging calls to write. Every trainer and evaluator in the remaining labs does the same, which is how you end up with 11 experiments you can compare side by side.

## About `ACCEPT_EULA`

Llama 3.2 is a gated model. Setting `ACCEPT_EULA = True` in the next cell records your acceptance of Meta's license — training will not start without it.

In [ ]:
ACCEPT_EULA = True

### Training configuration

The next three cells set up the training job:

1. **Config** — the base model (`meta-textgeneration-llama-3-2-3b-instruct`, the *instruct*-tuned variant, which already follows instructions and so gives SFT a much better starting point than the raw base) and the dataset ARNs Lab 1 registered in the AI Registry. Note you pass **ARNs, not file paths** — the trainer resolves the registered dataset version, so the exact data used is recorded and reproducible.
2. **`ModelPackageGroup`** — a versioned container in the SageMaker Model Registry. Every training run adds a new version to `sft-model`, and evaluation later looks up the most recent one.
3. **`SFTTrainer`** — ties it all together: model, data, LoRA, output location, role, and the MLflow destination.

In [ ]:
# Required Configs
BASE_MODEL = "meta-textgeneration-llama-3-2-3b-instruct"

# MODEL_PACKAGE_GROUP_NAME is same as CUSTOM_MODEL_NAME
MODEL_PACKAGE_GROUP_NAME = "sft-model"


# TRAINING_DATASET_ARN and VALIDATION_DATASET_ARN are stored by Lab 1
# (01-data-preparation.ipynb) and restored by the %store -r at the top of this
# notebook. If a NameError fires on either, run Lab 1 to completion first.
SFT_TRAINING_DATASET = TRAINING_DATASET_ARN

SFT_VALIDATION_DATASET = VALIDATION_DATASET_ARN

S3_OUTPUT_PATH = DEFAULT_BUCKET

ROLE_ARN = ROLE

In [ ]:
from botocore.exceptions import ClientError
from sagemaker.core.resources import ModelPackageGroup

# Create the versioned Model Registry container for the fine-tuned model.
# This is wrapped in try/except so the cell is safe to re-run: if you have
# already run this notebook (or restarted the kernel and re-executed the
# section), the group already exists and create() raises. In that case we
# fetch the existing group with get() -- both return a usable ModelPackageGroup
# object, which the SFTTrainer cell below consumes. This is the same idempotent
# pattern the SageMaker AI fine-tuning templates use.
try:
    model_package_group = ModelPackageGroup.create(
        model_package_group_name=MODEL_PACKAGE_GROUP_NAME,
        model_package_group_description='A fine-tuned text to sql model' #Required Description
    )
    print(f"Created model package group: {MODEL_PACKAGE_GROUP_NAME}")
except ClientError as e:
    if e.response['Error']['Code'] in ('ResourceInUse', 'ValidationException'):
        model_package_group = ModelPackageGroup.get(
            model_package_group_name=MODEL_PACKAGE_GROUP_NAME
        )
        print(f"Reusing existing model package group: {MODEL_PACKAGE_GROUP_NAME}")
    else:
        raise

print(f"  ARN: {model_package_group.model_package_group_arn}")

In [ ]:
from sagemaker.core.helper.session_helper import Session
sm_client = boto3.client("sagemaker", region_name=REGION)
# Create SageMaker session
sagemaker_session = Session(sagemaker_client=sm_client)

from sagemaker.train.sft_trainer import SFTTrainer
from sagemaker.train.common import TrainingType

trainer = SFTTrainer(
    model=BASE_MODEL,
    training_type=TrainingType.LORA,
    model_package_group=model_package_group,
    training_dataset=SFT_TRAINING_DATASET,
    s3_output_path=f's3://{S3_OUTPUT_PATH}/output',
    sagemaker_session=sagemaker_session,
    accept_eula=ACCEPT_EULA,
    role=ROLE,
    mlflow_resource_arn=MLFLOW_ARN,
    mlflow_experiment_name='sft-training',
)

### Override the hyperparameters

The trainer ships with defaults; this cell overrides the two that matter (`learning_rate` and `max_epochs`, reasoned about above) and prints the full resolved set.

**Read that printed dict.** It is the clearest single view of what SFT is about to do — batch size, sequence length, LoRA rank, warmup, scheduler. Everything not overridden here is a SageMaker default, and worth knowing about if you later want to tune this properly.

In [ ]:
from rich.pretty import pprint

# Modify options like object attributes
trainer.hyperparameters.learning_rate = 0.00001
trainer.hyperparameters.max_epochs = 8

print("Finetuning Hyperparameters:")
pprint(trainer.hyperparameters.to_dict())

### Run the training job

`trainer.train()` provisions a GPU instance, pulls the base model and your dataset, runs the 8 epochs, and registers the resulting adapter as a new version in the `sft-model` package group.

**This takes roughly 20-30 minutes**, most of it before any training starts — instance provisioning and downloading the base model weights dominate. The call blocks, streaming job logs into the cell output. A long quiet period early on is normal.

While you wait, there are two places to watch:

- **MLflow** → the `sft-training` experiment. The loss curve appears once training proper begins.
- **AWS Console** → SageMaker AI → Training → Training jobs, for infrastructure-level status.

### What a healthy run looks like

Training loss should fall steeply over the first two or three epochs — that is the model picking up your output format and schema, which is the easy part — and then flatten. By epoch 8 it should be low and fairly level.

If loss is still dropping sharply at epoch 8, more epochs might help. If it hits near-zero early and stays there, the model has memorized ~200 examples and further epochs are just deepening the overfit. Neither is a failure of the lab; both are the kind of read you would make on a real project.

The number that actually decides whether this worked, though, is the evaluation that follows — loss going down only tells you the model matched your examples more closely, not that it writes *correct* SQL. Those are different claims, which is exactly why you built the evaluator first.

In [ ]:
trainer.train()

# Evaluate the Fine-Tuned Model

Now the evaluator you built at the top of the notebook earns its keep.

`CustomScorerEvaluator` runs your fine-tuned model over a dataset, sends each generated query to the evaluator Lambda, and logs the four metrics to MLflow. Three arguments do the work:

| Argument | Why it matters |
|---|---|
| `model=model_package_arn` | Passing the **model package ARN** rather than the base model string is what targets your fine-tuned adapter. The next cell looks up the newest version in the `sft-model` group, which is the one you just trained |
| `evaluator=evaluator.arn` | Your Lambda. Same scorer used in Lab 3 for reward and Lab 4 for the frontier models |
| `evaluate_base_model=True` | **Do not remove this.** See below |

## Why `evaluate_base_model=True` matters

This runs the identical evaluation a second time against the **unmodified** Llama 3.2 3B, producing a run named `EvaluateBaseModel` alongside `EvaluateCustomModel`.

That gives you the before/after pair that makes the SFT result meaningful — "0.58 aggregate reward" means nothing until you know the untrained model scored 0.43 on the same 37 questions with the same scorer.

It is also the **only** place in the entire workshop that the base model is scored. Lab 3's evaluators deliberately pass `evaluate_base_model=False` (re-scoring an unchanged base model would just duplicate this row and burn a scarce concurrent evaluation slot), and Lab 4's comparison table pulls the `Llama 3.2 3B (base)` row from *this* experiment. Skip it here and that row is missing from your final results.

## Two datasets, two experiments

You evaluate against both splits, and the distinction is worth holding onto:

- **`validation.jsonl`** (~37 held-out samples, `sft-eval-validation`) — queries the model never trained on. This measures **generalization**, and it is the honest number.
- **`combined.jsonl`** (all ~241 samples, `sft-eval-combined`) — includes the training data, so it is partly a memorization test. It is here because it reflects something real: in production, most queries against a schema are repeats of patterns already seen. It answers "how well would this work on my actual traffic?" rather than "how well does it generalize?"

Both are legitimate; they answer different questions. Expect the combined number to be higher, and do not mistake that for a better result.

## Timing

Each evaluation takes roughly 10-15 minutes, and with `evaluate_base_model=True` each one is really two evaluations. Note also that **SageMaker allows only two evaluation jobs to run concurrently** — that limit does not bite in this lab, but it shapes how Lab 3 is sequenced.

In [ ]:
sm_client = boto3.client('sagemaker')
response = sm_client.list_model_packages(
    ModelPackageGroupName=MODEL_PACKAGE_GROUP_NAME,
    SortBy='CreationTime',
    SortOrder='Descending',
    MaxResults=1
)
model_package_arn = response['ModelPackageSummaryList'][0]['ModelPackageArn']
print(f"Using model package: {model_package_arn}")

In [ ]:
from sagemaker.train.evaluate import CustomScorerEvaluator

evaluation_job = CustomScorerEvaluator(
    evaluator=evaluator.arn,
    dataset=SFT_VALIDATION_DATASET,
    model=model_package_arn,
    s3_output_path=f's3://{S3_OUTPUT_PATH}/evaluation',
    mlflow_resource_arn=MLFLOW_ARN,
    mlflow_experiment_name='sft-eval-validation',
    model_package_group=model_package_group,
    evaluate_base_model=True,
)

In [ ]:
# Blocks until the validation evaluation pipeline reaches a terminal state.
evaluation_job.evaluate().wait()
print("Validation evaluation complete — base + SFT scores logged to "
      "the sft-eval-validation experiment in MLflow.")

## Evaluate Against Combined Set

Also evaluate against the full combined dataset to measure performance on the real query distribution (90% of production queries).

In [ ]:
# SFT_COMBINED_DATASET_ARN is stored by Lab 1 (01-data-preparation.ipynb) and
# restored by the %store -r at the top of this notebook. If a NameError fires
# on it, run Lab 1 to completion first.
combined_evaluation_job = CustomScorerEvaluator(
    evaluator=evaluator.arn,
    dataset=SFT_COMBINED_DATASET_ARN,
    model=model_package_arn,
    s3_output_path=f's3://{S3_OUTPUT_PATH}/evaluation-combined',
    mlflow_resource_arn=MLFLOW_ARN,
    mlflow_experiment_name='sft-eval-combined',
    model_package_group=model_package_group,
    evaluate_base_model=True,
)

combined_evaluation_job.evaluate().wait()
print("Combined evaluation complete — base + SFT scores logged to "
      "the sft-eval-combined experiment in MLflow.")

In [ ]:
# Persist for subsequent labs
EVALUATOR_ARN = evaluator.arn
%store EVALUATOR_ARN MODEL_PACKAGE_GROUP_NAME LAMBDA_FUNCTION_NAME

## Expected Results and Discussion

Your four evaluation runs are now in MLflow — `EvaluateBaseModel` and `EvaluateCustomModel` in each of `sft-eval-validation` and `sft-eval-combined`. This is the first real result of the workshop, so it is worth reading properly before moving on.

### What you should see on the validation set

Approximate values from a reference run. **Yours will differ** — see below.

| Model | Exec Success | Exact Match | Result F1 | Aggregate Reward |
|---|---|---|---|---|
| Llama 3.2 3B (base) | ~0.96 | ~0.19 | ~0.20 | ~0.43 |
| Llama 3.2 3B (SFT) | ~0.99 | ~0.37 | ~0.41 | ~0.58 |

### How to read that

**`execution_success` barely moves, and that is the point.** The base model already writes syntactically valid PostgreSQL — it was pre-trained on plenty of SQL. There was never anything to fix there. Note that a model scoring 0.96 on execution success and 0.20 on F1 is confidently producing well-formed queries that return the wrong rows, which is precisely the failure mode that makes text-to-SQL deceptive to evaluate.

**Result F1 roughly doubles, and that is the whole result.** The improvement is not "the model learned SQL". It learned *your* schema — which columns exist, what they are called, how they relate, what a question about "brand tier" or "customer segment" actually maps to. That is the claim the workshop makes, and this is the number that supports it.

**Exact Match stays low even after SFT (~0.37), and that is partly a measurement artifact.** Recall from the evaluator section that `execution_accuracy` is an ordered list comparison — a query that returns exactly the right rows in a different order scores zero. Treat Exact Match as a strict lower bound on correctness, and prefer Result F1 as the honest read.

**Aggregate reward goes from ~0.43 to ~0.58.** Remember the 0.30 floor: anything that parses gets 0.3 for free. So the meaningful range here is roughly 0.3-1.0, not 0-1.0, and moving from 0.43 to 0.58 covers a good fraction of the available headroom.

### Why your numbers will differ

- **Your dataset is your own.** The queries came from `pg_stat_statements` on your cluster and Bedrock wrote the natural language descriptions, so both the training data and the validation split differ from any reference run.
- **The validation split is small.** With ~37 samples, one query flipping from wrong to right moves Exact Match by about 0.03. Expect noise of that order.
- **Training is stochastic** — data ordering, LoRA initialization, GPU nondeterminism.

Judge this by whether the *pattern* holds — F1 and Exact Match up substantially, Exec Success roughly flat — not by matching numbers.

### What is still missing

SFT optimized for *resembling* your reference queries. Nothing in this lab optimized for being **correct** — the evaluator scored the model, but its scores never fed back into training. Loss went down because completions looked more like the references, which is related to correctness but is not the same thing.

That gap is what Lab 3 closes: RLVR takes this checkpoint, has it generate SQL, executes it, and trains on whether the results actually matched. Same Lambda, promoted from metric to reward.

The cell below prints a link straight to your MLflow tracking server. There is a fuller guide to reading these experiments in the **Reviewing Results** module at the end of the workshop.

In [ ]:
# Open MLflow to review the SFT results.
#
# Look at: sft-training (loss curve) and sft-eval-validation
# (EvaluateBaseModel vs EvaluateCustomModel).
from IPython.display import display, HTML

_sm = boto3.client("sagemaker", region_name=REGION)
try:
    _url = _sm.create_presigned_mlflow_app_url(Arn=MLFLOW_ARN)["AuthorizedUrl"]
    _label = "Open MLflow (signed link, valid ~5 minutes)"
except Exception as e:
    print(f"Presigned URL unavailable ({type(e).__name__}), falling back to console link.")
    _url = f"https://{REGION}.console.aws.amazon.com/sagemaker/home?region={REGION}#/mlflow"
    _label = "Open MLflow in the AWS Console"

display(HTML(f'<a href="{_url}" target="_blank"><b>{_label}</b></a>'))
print("\nExperiments written by this lab:")
for _e in ("sft-training", "sft-eval-validation", "sft-eval-combined"):
    print(f"  - {_e}")